# Module 5 - LiDAR-Based Urban Segmentation

In [ ]:
# Core scientific computing
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Geospatial processing
import rasterio
from rasterio.merge import merge
from rasterio.mask import mask as rasterio_mask
from rasterio.io import MemoryFile
import geopandas as gpd
from shapely.geometry import box

# Visualization
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, ListedColormap
from matplotlib.patches import Polygon as MPLPolygon
import folium
from folium import raster_layers
from pyproj import Transformer

# Image processing
from scipy import ndimage
from skimage.draw import polygon

# Deep learning
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

### 5.1. Load LiDAR Data


**KEY CONCEPTS:**
- **LiDAR** = Light Detection and Ranging, laser-based elevation measurement
- **DSM** = Digital Surface Model (includes buildings/trees)
- **DTM** = Digital Terrain Model (bare earth)
- **nDSM** = Normalized DSM (DSM - DTM = height above ground)
- https://remotesensingdata.gov.scot/data#/map
- https://www.youtube.com/watch?v=6BzPumsf2YA&t=1s

In [ ]:
# Define Edinburgh study area
lat1, lon1 = 55.94687, -3.19764
lat2, lon2 = 55.94204, -3.18954

# Create bounding box
min_lon, max_lon = min(lon1, lon2), max(lon1, lon2)
min_lat, max_lat = min(lat1, lat2), max(lat1, lat2)
bbox = box(min_lon, min_lat, max_lon, max_lat)

# Create GeoDataFrame
study_area = gpd.GeoDataFrame({'geometry': [bbox]}, crs='EPSG:4326')

print(f"✓ Study area defined: {(max_lat-min_lat)*111:.2f} km × {(max_lon-min_lon)*85:.2f} km")

In [ ]:
# File paths (downloaded from remotesensingdata.gov.scot)
dsm_paths = ["NT27SW_50CM_DSM_PHASE5.tif", "NT27SE_50CM_DSM_PHASE5.tif"]
dtm_paths = ["NT27SW_50CM_DTM_PHASE5.tif", "NT27SE_50CM_DTM_PHASE5.tif"]

# Merge DSM tiles
dsm_files = [rasterio.open(p) for p in dsm_paths]
dsm_mosaic, dsm_transform = merge(dsm_files)

# Merge DTM tiles
dtm_files = [rasterio.open(p) for p in dtm_paths]
dtm_mosaic, dtm_transform = merge(dtm_files)

# Get metadata
meta = dsm_files[0].meta.copy()
meta.update({
    "height": dsm_mosaic.shape[1],
    "width": dsm_mosaic.shape[2],
    "transform": dsm_transform
})

print(f"✓ Merged LiDAR tiles")
print(f"  Size: {meta['width']}×{meta['height']} pixels")
print(f"  Resolution: {dsm_files[0].res[0]:.2f} m/pixel")

#### Calculate nDSM


In [ ]:
# Calculate normalized DSM (height above ground)
ndsm = dsm_mosaic - dtm_mosaic

# Handle nodata values
nodata = meta.get("nodata", -9999)
ndsm = np.where((dsm_mosaic == nodata) | (dtm_mosaic == nodata), np.nan, ndsm)

print(f"✓ nDSM calculated")
print(f"  Height range: {np.nanmin(ndsm[0]):.1f} to {np.nanmax(ndsm[0]):.1f} m")
print(f"  nDSM = DSM - DTM (heights above ground)")

#### Clip to Study Area


In [ ]:
# Reproject study area to match raster CRS
study_area_crs = study_area.to_crs(meta["crs"])
bbox_geom = study_area_crs.union_all()

# Get bounding box
minx, miny, maxx, maxy = bbox_geom.bounds
clip_bbox = box(minx, miny, maxx, maxy)

# Clip nDSM to bounding box
with MemoryFile() as memfile:
    with memfile.open(**meta) as tmp:
        tmp.write(ndsm)
        clipped, clipped_transform = rasterio_mask(tmp, [clip_bbox.__geo_interface__], crop=True)


# Update metadata
meta_clipped = meta.copy()
meta_clipped.update({
    "height": clipped.shape[1],
    "width": clipped.shape[2],
    "transform": clipped_transform
})


# Save clipped nDSM
output_file = 'edinburgh_ndsm.tif'
with rasterio.open(output_file, 'w', **meta_clipped) as dst:
    dst.write(clipped)

print(f"✓ Clipped to study area: {clipped.shape[2]}×{clipped.shape[1]} pixels")
print(f"  Saved: {output_file}")

#### Visualize DSM, DTM, and nDSM

In [ ]:
# Clip DSM and DTM to same area for comparison
with MemoryFile() as memfile:
    with memfile.open(**meta) as tmp:
        tmp.write(dsm_mosaic)
        dsm_clipped, _ = rasterio_mask(tmp, [clip_bbox.__geo_interface__], crop=True)

with MemoryFile() as memfile:
    with memfile.open(**meta) as tmp:
        tmp.write(dtm_mosaic)
        dtm_clipped, _ = rasterio_mask(tmp, [clip_bbox.__geo_interface__], crop=True)

# Extract data arrays
dsm_data = dsm_clipped[0]
dtm_data = dtm_clipped[0]
ndsm_data = clipped[0]

In [ ]:
# Calculate percentile ranges for visualization
dsm_vmin, dsm_vmax = np.nanpercentile(dsm_data[~np.isnan(dsm_data)], [1, 99])
dtm_vmin, dtm_vmax = np.nanpercentile(dtm_data[~np.isnan(dtm_data)], [1, 99])
ndsm_vmax = np.nanpercentile(ndsm_data[~np.isnan(ndsm_data)], 99)

In [ ]:

# Visualize all three
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(dsm_data, cmap='terrain', vmin=dsm_vmin, vmax=dsm_vmax)
axes[0].set_title('DSM (Surface + Terrain)', fontsize=12, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(dtm_data, cmap='terrain', vmin=dtm_vmin, vmax=dtm_vmax)
axes[1].set_title('DTM (Bare Earth)', fontsize=12, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(ndsm_data, cmap='hot', vmin=0, vmax=ndsm_vmax)
axes[2].set_title('nDSM = DSM - DTM\n(Object Heights)', fontsize=12, fontweight='bold')
axes[2].axis('off')

plt.suptitle('LiDAR Products: Edinburgh Study Area', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


#### Hillshade Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Raw nDSM with height colormap
axes[0].imshow(ndsm_data, cmap='hot', vmin=0, vmax=ndsm_vmax)
axes[0].set_title('nDSM (Height Above Ground)', fontsize=13, fontweight='bold')
axes[0].axis('off')

# Hillshade showing 3D structure
ls = LightSource(azdeg=315, altdeg=45)
hillshade = ls.hillshade(ndsm_data, vert_exag=3, dx=0.5, dy=0.5)
axes[1].imshow(hillshade, cmap='gray')
axes[1].set_title('Hillshade (3D Terrain)', fontsize=13, fontweight='bold')
axes[1].axis('off')

plt.suptitle('Edinburgh - Building and Tree Heights', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 2. Engineer Features


**KEY CONCEPTS:**
- **Texture** = surface roughness measured as local standard deviation
- **Local standard deviation** = height variability within a small neighborhood


#### Calculate Texture

In [ ]:
# Calculate surface texture (roughness) from nDSM
window_size = 3  # 3×3 pixel neighborhood

# Replace NaN with 0 for processing
ndsm_clean = np.nan_to_num(ndsm_data, nan=0)

# Calculate local mean and variance using convolution filters
local_mean = ndimage.uniform_filter(ndsm_clean, size=window_size)
local_mean_sq = ndimage.uniform_filter(ndsm_clean**2, size=window_size)
local_variance = local_mean_sq - local_mean**2
local_variance[local_variance < 0] = 0  # Numerical stability


# Texture = standard deviation (sqrt of variance)
texture = np.sqrt(local_variance)

# Restore NaN in original nodata locations
texture[np.isnan(ndsm_data)] = np.nan

print(f"✓ Texture calculated")
print(f"  Range: {np.nanmin(texture):.3f} to {np.nanmax(texture):.3f}")
print(f"  Low values = smooth (buildings), High values = rough (trees)")

####  Visualize Two-Channel Input

In [ ]:
# Visualize height and texture side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Channel 1: Height (nDSM)
axes[0].imshow(ndsm_data, cmap='hot', vmin=0, vmax=30)
axes[0].set_title('Channel 1: Height (m)\nTall = Buildings OR Trees', 
                  fontweight='bold', fontsize=12)
axes[0].axis('off')

# Channel 2: Texture (roughness)
axes[1].imshow(texture, cmap='gray', vmin=0, vmax=2)
axes[1].set_title('Channel 2: Texture (roughness)\nDark = Smooth (Buildings), Bright = Rough (Trees)', 
                  fontweight='bold', fontsize=12)
axes[1].axis('off')

plt.suptitle('Two-Channel Input for U-Net: Height + Texture', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Combine height and texture into 2-channel input array
input_2channel = np.stack([ndsm_data, texture], axis=0)  # Shape: (2, H, W)

### 5.3. Create Training Labels

**KEY CONCEPTS:**
- **Manual labeling** = human-drawn polygons defining building and tree regions
- **Interactive labeling tool** = matplotlib interface for polygon digitization
- **Height-based cleaning** = refining polygon labels using elevation thresholds
- **Unlabeled pixels (255)** = areas we're uncertain about, excluded from training
- **Three classes** = Ground (0), Building (1), Tree (2)


#### Load nDSM

In [ ]:
# Load the nDSM we created earlier
with rasterio.open('edinburgh_ndsm.tif') as src:
    ndsm = src.read(1)
    ndsm_meta = src.meta
    ndsm_transform = src.transform

# Clean nodata values
ndsm = np.where(ndsm < -100, np.nan, ndsm)

print(f"✓ nDSM loaded for labeling")
print(f"  Size: {ndsm.shape}")
print(f"  Height range: {np.nanmin(ndsm):.1f} to {np.nanmax(ndsm):.1f} m")

#### Interactive Labeling Tool 


In [ ]:


matplotlib.use('TkAgg')  # Interactive backend

class ManualLabeler:
    """Interactive polygon labeling tool"""
    def __init__(self, ndsm):
        # Extract 2D array if 3D
        if ndsm.ndim == 3:
            ndsm = ndsm[0]
        
        self.ndsm = ndsm
        self.labels = np.zeros(ndsm.shape, dtype=np.uint8)
        self.current_points = []
        self.current_class = 1  # 1=Building, 2=Tree
        self.polygons_drawn = 0
        
        # Create interactive plot
        plt.ion()
        self.fig, self.ax = plt.subplots(figsize=(14, 14))
        self.ax.imshow(ndsm, cmap='hot', vmin=0, vmax=30)
        self.ax.set_title('CLICK=point | B=building | T=tree | ENTER=finish | Q=quit',
                         fontsize=13, fontweight='bold')
        self.ax.axis('off')
        
        # Connect mouse and keyboard
        self.fig.canvas.mpl_connect('button_press_event', self.onclick)
        self.fig.canvas.mpl_connect('key_press_event', self.onkey)
    
    def onclick(self, event):
        """Add point to polygon"""
        if event.inaxes and event.button == 1:
            x, y = int(event.xdata), int(event.ydata)
            self.current_points.append([x, y])
            self.ax.plot(x, y, 'yo', markersize=8)
            
            if len(self.current_points) > 1:
                pts = np.array(self.current_points)
                self.ax.plot(pts[-2:, 0], pts[-2:, 1], 'y-', linewidth=2)
            plt.draw()
    
    def onkey(self, event):
        """Handle keyboard shortcuts"""
        if event.key == 'b':
            self.current_class = 1
            print("→ Building mode")
        elif event.key == 't':
            self.current_class = 2
            print("→ Tree mode")
        elif event.key == 'enter' and len(self.current_points) > 2:
            points = np.array(self.current_points)
            rr, cc = polygon(points[:, 1], points[:, 0], self.labels.shape)
            self.labels[rr, cc] = self.current_class
            
            color = 'red' if self.current_class == 1 else 'green'
            poly = MPLPolygon(points, closed=True, fill=True,
                            facecolor=color, alpha=0.4)
            self.ax.add_patch(poly)
            plt.draw()
            
            self.polygons_drawn += 1
            self.current_points = []
            print(f"  ✓ Polygon {self.polygons_drawn} saved")
        
        elif event.key == 'q':
            plt.close(self.fig)

# Interactive labeling (uncomment to use)
labeler = ManualLabeler(ndsm)
plt.show(block=True)
labels_raw = labeler.labels

# Save raw labels
with rasterio.open('edinburgh_manual_labels.tif', 'w',
                   driver='GTiff',
                   height=labels_raw.shape[0],
                   width=labels_raw.shape[1],
                   count=1,
                   dtype=np.uint8,
                   crs=ndsm_meta['crs'],
                   transform=ndsm_transform) as dst:
    dst.write(labels_raw, 1)

print(f"✓ Saved raw labels: edinburgh_manual_labels.tif")
print(f"  Polygons drawn: {labeler.polygons_drawn}")

#### Load Pre-Made Labels

In [ ]:
# Load pre-labeled data (skip interactive labeling for demo)
with rasterio.open('edinburgh_manual_labels_preload.tif') as src:
    labels_raw = src.read(1)

print(f"✓ Pre-made labels loaded")
print(f"  Buildings marked: {np.sum(labels_raw == 1):,} pixels")
print(f"  Trees marked:     {np.sum(labels_raw == 2):,} pixels")
print(f"  Background:       {np.sum(labels_raw == 0):,} pixels")

#### Clean Labels with Height Thresholds

In [ ]:
# Ensure nDSM and labels have matching shapes
if ndsm.shape != labels_raw.shape:
    print(f"⚠️  Shape mismatch detected!")
    print(f"  nDSM:   {ndsm.shape}")
    print(f"  Labels: {labels_raw.shape}")
    
    # Crop both to minimum common size
    min_h = min(ndsm.shape[0], labels_raw.shape[0])
    min_w = min(ndsm.shape[1], labels_raw.shape[1])
    
    ndsm = ndsm[:min_h, :min_w]
    labels_raw = labels_raw[:min_h, :min_w]
    
    print(f"  ✓ Cropped both to: {ndsm.shape}")

# Initialize all pixels as unlabeled (255 = ignored in training)
labels_cleaned = np.full_like(labels_raw, 255, dtype=np.uint8)

In [ ]:
# Extract polygon masks from manual labels
building_polygons = (labels_raw == 1)
tree_polygons = (labels_raw == 2)
labeled_area = (building_polygons | tree_polygons)

# Apply physics-based height rules within labeled polygons only
labels_cleaned[labeled_area & (ndsm < 1.0)] = 0  # Ground
labels_cleaned[building_polygons & (ndsm >= 5.0)] = 1  # Buildings
labels_cleaned[tree_polygons & (ndsm >= 2.0)] = 2  # Trees


print(f"\n✓ Labels cleaned with height-based rules")
print(f"  Ground (0):      {np.sum(labels_cleaned == 0):,} pixels (<1m)")
print(f"  Buildings (1):   {np.sum(labels_cleaned == 1):,} pixels (≥5m)")
print(f"  Trees (2):       {np.sum(labels_cleaned == 2):,} pixels (≥2m)")
print(f"  Unlabeled (255): {np.sum(labels_cleaned == 255):,} pixels (ignored)")

#### Visualize Cleaned Labels

In [ ]:
# Switch matplotlib back to inline mode for notebook
import matplotlib
%matplotlib inline


# Prepare labels for visualization (map 255 to black)
display_labels = labels_cleaned.copy()
display_labels[display_labels == 255] = 3  # Temporary value for black color


# 4-color map: brown=ground, red=building, green=tree, black=unlabeled
cmap_labels = ListedColormap(['#8B4513', '#FF0000', '#00FF00', '#000000'])
cmap_raw = ListedColormap(['#8B4513', '#FF0000', '#00FF00'])



fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# Raw manual polygons
axes[0].imshow(labels_raw, cmap=cmap_raw, vmin=0, vmax=2)
axes[0].set_title('Raw Polygons\n(Manual Labels)', fontweight='bold', fontsize=12)
axes[0].axis('off')


# Height data (nDSM)
axes[1].imshow(ndsm, cmap='hot', vmin=0, vmax=30)
axes[1].set_title('Height (nDSM)\n(Physics)', fontweight='bold', fontsize=12)
axes[1].axis('off')

# Cleaned labels
axes[2].imshow(display_labels, cmap=cmap_labels, vmin=0, vmax=3)
axes[2].set_title(f'Cleaned Labels\nG:{np.sum(labels_cleaned==0):,} B:{np.sum(labels_cleaned==1):,} T:{np.sum(labels_cleaned==2):,}',
                  fontweight='bold', fontsize=12)
axes[2].axis('off')

plt.suptitle('Manual + Physics = Clean Ground Truth | Brown=Ground | Red=Building | Green=Tree | Black=Unlabeled', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()



# save this
with rasterio.open('edinburgh_manual_labels_cleaned.tif', 'w',
                   driver='GTiff',
                   height=labels_cleaned.shape[0],
                   width=labels_cleaned.shape[1],
                   count=1,
                   dtype=np.uint8,
                   crs=ndsm_meta['crs'],
                   transform=ndsm_transform) as dst:
    dst.write(labels_cleaned, 1)

### 5.4. Prepare Training Data with Spatial Split

**KEY CONCEPTS:**
- **Spatial split** = geographic separation (left vs right) prevents data leakage
- **Random split within region** = 80/20 split of left-half patches for train/val
- **Balanced sampling** = ensuring adequate representation of all three classes
- **Multi-class segmentation** = 3 output classes (ground/building/tree) vs binary
- **Unlabeled masking** = pixels with label=255 (black) ignored in loss calculation
- **Patch normalization** = standardizing each patch to zero mean, unit variance


#### Load Clean Labels

In [ ]:
# Load cleaned labels
with rasterio.open('edinburgh_manual_labels_cleaned.tif') as src:
    labels = src.read(1)


# Prepare for visualization
display_labels = labels.copy()
display_labels[display_labels == 255] = 3  # Map unlabeled to black

# Visualize the full scene
cmap_labels = ListedColormap(['#8B4513', '#FF0000', '#00FF00', '#000000'])



fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Height
axes[0].imshow(ndsm, cmap='hot', vmin=0, vmax=30)
axes[0].set_title('Height (nDSM)', fontweight='bold', fontsize=12)
axes[0].axis('off')

# Texture
axes[1].imshow(texture, cmap='gray', vmin=0, vmax=2)
axes[1].set_title('Texture (Roughness)', fontweight='bold', fontsize=12)
axes[1].axis('off')

# Labels
axes[2].imshow(display_labels, cmap=cmap_labels, vmin=0, vmax=3)
axes[2].set_title(f'Labels\nG:{np.sum(labels==0):,} B:{np.sum(labels==1):,} T:{np.sum(labels==2):,}',
                  fontweight='bold', fontsize=12)
axes[2].axis('off')

plt.suptitle('Full Scene: Input Channels + Ground Truth', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print(f"✓ Loaded clean labels: {labels.shape}")
print(f"  Ground (0):      {np.sum(labels==0):,} pixels")
print(f"  Buildings (1):   {np.sum(labels==1):,} pixels")
print(f"  Trees (2):       {np.sum(labels==2):,} pixels")
print(f"  Unlabeled (255): {np.sum(labels==255):,} pixels")

#### Spatial Split


In [ ]:
# Spatial split - left half = train/val region, right half = final test region
h, w = ndsm.shape
split_col = w // 2

# Split height (nDSM)
ndsm_train = ndsm[:, :split_col]
ndsm_test = ndsm[:, split_col:]

# Split texture
texture_train = texture[:, :split_col]
texture_test = texture[:, split_col:]

# Split labels
labels_train = labels[:, :split_col]
labels_test = labels[:, split_col:]


print(f"✓ Spatial split applied")
print(f"  Left region:  {ndsm_train.shape} (will extract patches for train/val)")
print(f"  Right region: {ndsm_test.shape} (reserved for final test)")

In [ ]:
# Visualize spatial split
display_train = labels_train.copy()
display_train[display_train == 255] = 3
display_test = labels_test.copy()
display_test[display_test == 255] = 3

fig, axes = plt.subplots(2, 3, figsize=(12, 10))



# === LEFT REGION (TRAIN/VAL) ===
axes[0, 0].imshow(ndsm_train, cmap='hot', vmin=0, vmax=30)
axes[0, 0].set_title('Left - Height', fontweight='bold', fontsize=11, color='blue')
axes[0, 0].axis('off')

axes[0, 1].imshow(texture_train, cmap='gray', vmin=0, vmax=2)
axes[0, 1].set_title('Left - Texture', fontweight='bold', fontsize=11, color='blue')
axes[0, 1].axis('off')

axes[0, 2].imshow(display_train, cmap=cmap_labels, vmin=0, vmax=3)
axes[0, 2].set_title(f'Left - Labels\nG:{np.sum(labels_train==0):,} B:{np.sum(labels_train==1):,} T:{np.sum(labels_train==2):,}',
                     fontweight='bold', fontsize=11, color='blue')
axes[0, 2].axis('off')



# === RIGHT REGION (FINAL TEST) ===
axes[1, 0].imshow(ndsm_test, cmap='hot', vmin=0, vmax=30)
axes[1, 0].set_title('Right - Height (Reserved)', fontweight='bold', fontsize=11, color='red')
axes[1, 0].axis('off')

axes[1, 1].imshow(texture_test, cmap='gray', vmin=0, vmax=2)
axes[1, 1].set_title('Right - Texture (Reserved)', fontweight='bold', fontsize=11, color='red')
axes[1, 1].axis('off')

axes[1, 2].imshow(display_test, cmap=cmap_labels, vmin=0, vmax=3)
axes[1, 2].set_title(f'Right - Labels (Reserved)\nG:{np.sum(labels_test==0):,} B:{np.sum(labels_test==1):,} T:{np.sum(labels_test==2):,}',
                     fontweight='bold', fontsize=11, color='red')
axes[1, 2].axis('off')

plt.suptitle('Spatial Split: Left (Train/Val) vs Right (Final Test)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

#### Extract Patches from Left Region


In [ ]:
def extract_balanced_patches_with_ground(ndsm, texture, labels, patch_size=128, stride=64):
    """
    Extract patches with balanced class representation.
    Returns patches with 2 channels: height (nDSM) + texture
    """
    h, w = ndsm.shape
    
    building_patches = []
    tree_patches = []
    ground_patches = []
    mixed_patches = []

    for i in range(0, h - patch_size + 1, stride):
        for j in range(0, w - patch_size + 1, stride):
            # Extract patch from all inputs
            ndsm_p = ndsm[i:i+patch_size, j:j+patch_size]
            texture_p = texture[i:i+patch_size, j:j+patch_size]
            label_p = labels[i:i+patch_size, j:j+patch_size]

            # Skip patches with very low variance (empty areas)
            if np.nanstd(ndsm_p) < 0.5:
                continue

            # Count pixels per class
            n_ground = np.sum(label_p == 0)
            n_buildings = np.sum(label_p == 1)
            n_trees = np.sum(label_p == 2)
            n_unlabeled = np.sum(label_p == 255)
            n_labeled_total = n_ground + n_buildings + n_trees

            # Skip patches with too many unlabeled pixels (>80%)
            if n_unlabeled > (patch_size * patch_size * 0.8):
                continue
            
            # Skip patches with very few labeled pixels
            if n_labeled_total < 100:
                continue

            # Store as tuple (ndsm, texture, labels)
            patch_tuple = (ndsm_p, texture_p, label_p)     

            # Categorize patches by dominant class
            if n_buildings > 500 and n_trees > 200:
                mixed_patches.append(patch_tuple)  # Best: has both
            elif n_buildings > 500:
                building_patches.append(patch_tuple)
            elif n_trees > 200:
                tree_patches.append(patch_tuple)
            elif n_ground > 500:
                ground_patches.append(patch_tuple)

    print(f"  Extracted patches:")
    print(f"    Building-dominant: {len(building_patches)}")
    print(f"    Tree-dominant:     {len(tree_patches)}")
    print(f"    Ground-dominant:   {len(ground_patches)}")
    print(f"    Mixed (B+T):       {len(mixed_patches)}")

    # Balanced sampling
    import random
    random.seed(42)
    
    # Oversample trees (usually fewer), match buildings, limit ground
    selected_trees = tree_patches * 2
    selected_buildings = random.sample(building_patches, 
                                      min(len(tree_patches), len(building_patches)))
    selected_ground = random.sample(ground_patches, 
                                   min(20, len(ground_patches)))

    # Combine all selected patches
    all_patches = selected_buildings + selected_trees + mixed_patches + selected_ground
    
    # Separate into individual lists
    ndsm_patches = [p[0] for p in all_patches]
    texture_patches = [p[1] for p in all_patches]
    label_patches = [p[2] for p in all_patches]
    
    # Print final statistics
    total_ground = sum(np.sum(p == 0) for p in label_patches)
    total_buildings = sum(np.sum(p == 1) for p in label_patches)
    total_trees = sum(np.sum(p == 2) for p in label_patches)
    total_unlabeled = sum(np.sum(p == 255) for p in label_patches)
    
    print(f"  Balanced selection: {len(ndsm_patches)} patches")
    print(f"    Ground pixels:    {total_ground:,}")
    print(f"    Building pixels:  {total_buildings:,}")
    print(f"    Tree pixels:      {total_trees:,}")
    print(f"    Unlabeled pixels: {total_unlabeled:,} (ignored)")
    
    return ndsm_patches, texture_patches, label_patches

In [ ]:
# Extract ONLY from left half (will be split 80/20 for train/val)
print("Extracting patches from LEFT half:")
ndsm_patches_left, texture_patches_left, label_patches_left = \
    extract_balanced_patches_with_ground(ndsm_train, texture_train, labels_train, 
                                        patch_size=128, stride=64)


print(f"\n✓ Extraction complete")
print(f"  Left-half patches: {len(ndsm_patches_left)}")
print(f"  Will split 80/20 for training and validation")
print(f"  Right half reserved for final testing after training completes")

#### Normalize Patches


In [ ]:
# Normalize patches (zero mean, unit variance)
def normalize_patches(patches):
    """Normalize each patch to zero mean, unit variance"""
    return [np.nan_to_num((p - np.nanmean(p)) / (np.nanstd(p) + 1e-8), 0) 
            for p in patches]

# Normalize left-half patches
ndsm_left_norm = normalize_patches(ndsm_patches_left)
texture_left_norm = normalize_patches(texture_patches_left)

print(f"✓ Patches normalized")
print(f"  Left half: {len(ndsm_left_norm)} patches × 2 channels")
print(f"  Ready for 80/20 train/val split")

In [ ]:
%matplotlib inline

# Colormap for labels
cmap_labels = ListedColormap(['#8B4513', '#FF0000', '#00FF00', '#000000'])

# Select 6 random patches from left half
sample_idx = np.random.choice(len(ndsm_patches_left), 6, replace=False)

fig, axes = plt.subplots(6, 2, figsize=(10, 20))

for i, idx in enumerate(sample_idx):
    # Left: Height (nDSM)
    axes[i, 0].imshow(ndsm_patches_left[idx], cmap='hot', vmin=0, vmax=30)
    axes[i, 0].set_title(f'Patch {idx} - Height', fontsize=10, fontweight='bold')
    axes[i, 0].axis('off')
    
    # Right: Texture + Labels overlay
    axes[i, 1].imshow(texture_patches_left[idx], cmap='gray', vmin=0, vmax=2)
    
    # Overlay labels (map 255 to 3 for visualization)
    label_display = label_patches_left[idx].copy()
    label_display[label_display == 255] = 3
    label_overlay = np.ma.masked_where(label_patches_left[idx] == 255, label_display)
    axes[i, 1].imshow(label_overlay, cmap=cmap_labels, alpha=0.5, vmin=0, vmax=3)
    
    # Count pixels per class
    n_g = np.sum(label_patches_left[idx] == 0)
    n_b = np.sum(label_patches_left[idx] == 1)
    n_t = np.sum(label_patches_left[idx] == 2)
    
    axes[i, 1].set_title(f'Texture + Labels\nG:{n_g} B:{n_b} T:{n_t}', 
                         fontsize=10, fontweight='bold')
    axes[i, 1].axis('off')

plt.suptitle('Left-Half Patches: Height (left) + Texture+Labels (right)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Showing 6 sample patches from {len(ndsm_patches_left)} total (left half)")
print("Brown=Ground | Red=Building | Green=Tree | Black=Unlabeled")

#### Create PyTorch Datasets

In [ ]:
class LiDARDataset(Dataset):
    """PyTorch Dataset for 2-channel LiDAR input + 3-class labels"""
    
    def __init__(self, ndsm_patches, texture_patches, label_patches):
        self.ndsm = ndsm_patches
        self.texture = texture_patches
        self.labels = label_patches
    
    def __len__(self):
        return len(self.ndsm)
    
    def __getitem__(self, idx):
        # Stack height + texture as 2-channel input (2, H, W)
        ndsm_t = torch.FloatTensor(self.ndsm[idx])
        texture_t = torch.FloatTensor(self.texture[idx])
        input_t = torch.stack([ndsm_t, texture_t], dim=0)
        
        # Labels as LongTensor for CrossEntropyLoss
        labels_t = torch.LongTensor(self.labels[idx])
        
        return input_t, labels_t

# Create full dataset from left-half patches, then random 80/20 split
full_dataset = LiDARDataset(ndsm_left_norm, texture_left_norm, label_patches_left)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size


train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)


print(f"✓ PyTorch datasets created")
print(f"  Training:   {len(train_dataset)} patches (80% of left half)")
print(f"  Validation: {len(val_dataset)} patches (20% of left half)")
print(f"  Input:      (2, 128, 128) - height + texture")
print(f"  Labels:     (128, 128) - 3 classes + unlabeled (255)")
print(f"\n✅ Ready for U-Net training!")
print(f"  Right half will be evaluated via sliding window after training")

### 5.5. Train U-Net Model

**KEY CONCEPTS:**
- **Skip connections** = encoder features concatenated to decoder (`torch.cat([d3, e3])`)
- **BatchNorm2d** = normalizes activations for stable, faster training
- **Multi-class segmentation** = 3 output channels (ground/building/tree)
- **CrossEntropyLoss** = multi-class loss function vs BCELoss (binary)
- **Class weights** = compensate for imbalanced classes (more ground than trees)
- **ignore_index=255** = unlabeled pixels excluded from loss calculation


In [ ]:
class UNet(nn.Module):
    def __init__(self, in_channels=2, num_classes=3):
        super(UNet, self).__init__()
        
        # Encoder (downsampling)
        self.enc1 = self.conv_block(in_channels, 16)
        self.enc2 = self.conv_block(16, 32)
        self.enc3 = self.conv_block(32, 64)
        
        # Bottleneck
        self.bottleneck = self.conv_block(64, 128)
        
        # Decoder (upsampling) with skip connections
        self.upconv3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = self.conv_block(128, 64)  # 128 = 64 + 64 from skip
        
        self.upconv2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = self.conv_block(64, 32)   # 64 = 32 + 32 from skip
        
        self.upconv1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = self.conv_block(32, 16)   # 32 = 16 + 16 from skip
        
        # Output: 3 classes
        self.out = nn.Conv2d(16, num_classes, 1)
    
    def conv_block(self, in_ch, out_ch):
        """Double convolution with BatchNorm"""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        # Encoder: save features for skip connections
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        
        # Bottleneck
        b = self.bottleneck(F.max_pool2d(e3, 2))
        
        # Decoder: concatenate skip connections
        d3 = self.upconv3(b)
        d3 = torch.cat([d3, e3], dim=1)  # Skip connection
        d3 = self.dec3(d3)
        
        d2 = self.upconv2(d3)
        d2 = torch.cat([d2, e2], dim=1)  # Skip connection
        d2 = self.dec2(d2)
        
        d1 = self.upconv1(d2)
        d1 = torch.cat([d1, e1], dim=1)  # Skip connection
        d1 = self.dec1(d1)
        
        return self.out(d1)  # (B, 3, H, W) - 3 class probabilities


# Create model
model = UNet(in_channels=2, num_classes=3)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)


total_params = sum(p.numel() for p in model.parameters())
print(f"✓ Full U-Net created")
print(f"  Input: 2 channels (height + texture)")
print(f"  Output: 3 classes (ground/building/tree)")
print(f"  Parameters: {total_params:,}")
print(f"  Device: {device}")

#### Setup Training


In [ ]:
BATCH_SIZE = 8
NUM_EPOCHS = 30

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# Calculate class weights (inverse frequency)
# From our training data: Ground most common, trees least common
class_weights = torch.FloatTensor([0.5, 1.5, 1.5]).to(device)  # [ground, building, tree]

# Loss: CrossEntropyLoss with weights and ignore unlabeled (255)
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=255)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



print(f"✓ Training setup")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Loss: CrossEntropyLoss (3-class)")
print(f"  Class weights: Ground={class_weights[0]:.1f}, Building={class_weights[1]:.1f}, Tree={class_weights[2]:.1f}")
print(f"  Ignoring unlabeled pixels (255)")
print(f"  Training batches:   {len(train_loader)} (80% of left half)")
print(f"  Validation batches: {len(val_loader)} (20% of left half)")

#### Training Loop

In [ ]:
# Training tracking
train_losses = []
val_losses = []
val_accs = []

# Early stopping
best_val_loss = float('inf')
patience = 10
patience_counter = 0

print(f"Training for {NUM_EPOCHS} epochs on {device}...")
print("="*60)


for epoch in range(NUM_EPOCHS):
    # === TRAINING PHASE ===
    model.train()
    train_loss = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    train_loss /= len(train_loader)
    train_losses.append(train_loss)


    # === VALIDATION PHASE (LEFT HALF - 20%) ===
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Accuracy on labeled pixels only (ignore 255)
            preds = torch.argmax(outputs, dim=1)
            valid_mask = (labels != 255)
            correct += ((preds == labels) & valid_mask).sum().item()
            total += valid_mask.sum().item()

    val_loss /= len(val_loader)
    val_acc = correct / total
    
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Early stopping based on validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'unet_lidar_best.pth')
        status = "⭐ BEST"
    else:
        patience_counter += 1
        status = f"({patience_counter}/{patience})"
    
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.4f} {status}")
    
    if patience_counter >= patience:
        print(f"\n⚠️ Early stopping - no improvement for {patience} epochs")
        break


print("\n" + "="*60)
print(f"✓ Training complete!")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Model saved: unet_lidar_best.pth")

#### Visualize Training Progress


In [ ]:
%matplotlib inline

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(train_losses, label='Train (Left Half - 80%)', linewidth=2, marker='o')
axes[0].plot(val_losses, label='Validation (Left Half - 20%)', linewidth=2, marker='s')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Progress - Loss', fontweight='bold', fontsize=13)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(val_accs, linewidth=2, color='green', marker='o')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Validation Accuracy (Left Half - 20%)', fontweight='bold', fontsize=13)
axes[1].grid(alpha=0.3)

plt.suptitle(f'U-Net Training | Best Val Loss: {best_val_loss:.4f}', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n✅ Final validation accuracy: {val_accs[-1]:.4f}")
print(f"   Right half test set still untouched — evaluated next.")

### 5.6. Evaluate and Visualize Results


#### Load Best Model and Predict

In [ ]:
# Load best model
model.load_state_dict(torch.load('unet_lidar_best.pth'))
model.eval()

print("Running sliding window inference on RIGHT half (unseen test region)...")

In [ ]:
# Initialize prediction map
patch_size = 128
stride = 64
pred_map_right = np.zeros(ndsm_test.shape, dtype=np.float32)
count_map = np.zeros(ndsm_test.shape, dtype=np.float32)

with torch.no_grad():
    for i in range(0, ndsm_test.shape[0] - patch_size + 1, stride):
        for j in range(0, ndsm_test.shape[1] - patch_size + 1, stride):
            # Extract and normalize patch
            ndsm_patch = ndsm_test[i:i+patch_size, j:j+patch_size]
            texture_patch = texture_test[i:i+patch_size, j:j+patch_size]
            
            ndsm_norm = np.nan_to_num((ndsm_patch - np.nanmean(ndsm_patch)) / (np.nanstd(ndsm_patch) + 1e-8), 0)
            texture_norm = np.nan_to_num((texture_patch - np.nanmean(texture_patch)) / (np.nanstd(texture_patch) + 1e-8), 0)
            
            # Create input tensor
            input_t = torch.from_numpy(
                np.stack([ndsm_norm, texture_norm], axis=0)
            ).unsqueeze(0).float().to(device)
            
            # Predict
            output = model(input_t)
            pred = output.argmax(dim=1).squeeze().cpu().numpy()
            
            # Accumulate
            pred_map_right[i:i+patch_size, j:j+patch_size] += pred
            count_map[i:i+patch_size, j:j+patch_size] += 1
        
        if i % 256 == 0:
            print(f"  Progress: {i}/{ndsm_test.shape[0]}", end='\r')

# Average overlapping predictions
pred_map_right = (pred_map_right / (count_map + 1e-8)).astype(np.uint8)

print(f"\n✓ Right-half inference complete")
print(f"  Shape: {pred_map_right.shape}")
print(f"  Coverage: {(count_map > 0).sum() / count_map.size * 100:.1f}%")

In [ ]:
# Run sliding window inference on LEFT half (training region)
print("Running full inference on LEFT-half region (training)...")

pred_map_left = np.zeros(ndsm_train.shape, dtype=np.float32)
count_map_left = np.zeros(ndsm_train.shape, dtype=np.float32)

model.eval()
patch_size = 128
stride = 64

with torch.no_grad():
    for i in range(0, ndsm_train.shape[0] - patch_size + 1, stride):
        for j in range(0, ndsm_train.shape[1] - patch_size + 1, stride):
            # Extract patch
            ndsm_patch = ndsm_train[i:i+patch_size, j:j+patch_size]
            texture_patch = texture_train[i:i+patch_size, j:j+patch_size]
            
            # Normalize
            ndsm_norm = np.nan_to_num((ndsm_patch - np.nanmean(ndsm_patch)) / (np.nanstd(ndsm_patch) + 1e-8), 0)
            texture_norm = np.nan_to_num((texture_patch - np.nanmean(texture_patch)) / (np.nanstd(texture_patch) + 1e-8), 0)
            
            # Create tensor
            input_t = torch.from_numpy(
                np.stack([ndsm_norm, texture_norm], axis=0)
            ).unsqueeze(0).float().to(device)
            
            # Predict
            output = model(input_t)
            pred = output.argmax(dim=1).squeeze().cpu().numpy()
            
            # Accumulate
            pred_map_left[i:i+patch_size, j:j+patch_size] += pred
            count_map_left[i:i+patch_size, j:j+patch_size] += 1
        
        if i % 256 == 0:
            print(f"  Progress: {i}/{ndsm_train.shape[0]}", end='\r')

# Average overlapping predictions
pred_map_left = (pred_map_left / (count_map_left + 1e-8)).astype(np.uint8)

#### Calculate Per-Class IoU


In [ ]:
def calculate_iou_per_class(predictions, labels, num_classes=3, ignore_index=255):
    """Calculate IoU for each class separately"""
    ious = []
    
    for cls in range(num_classes):
        # Binary masks for this class
        pred_mask = (predictions == cls)
        true_mask = (labels == cls)
        
        # Exclude unlabeled pixels
        valid_mask = (labels != ignore_index)
        
        # Intersection and union
        intersection = ((pred_mask & true_mask) & valid_mask).sum()
        union = ((pred_mask | true_mask) & valid_mask).sum()
        
        # IoU
        iou = (intersection + 1e-6) / (union + 1e-6)
        ious.append(iou)
    
    return ious

# Calculate IoU on LEFT half (trained region)
class_ious_left = calculate_iou_per_class(pred_map_left, labels_train)

# Calculate IoU on RIGHT half (unseen test region)
class_ious_right = calculate_iou_per_class(pred_map_right, labels_test)

print("="*60)
print("PER-CLASS IoU RESULTS")
print("="*60)
print("\nLEFT HALF (Trained Region):")
print(f"  Ground (0):    {class_ious_left[0]:.4f}")
print(f"  Buildings (1): {class_ious_left[1]:.4f}")
print(f"  Trees (2):     {class_ious_left[2]:.4f}")
print(f"  Mean IoU:      {np.mean(class_ious_left):.4f}")

print("\nRIGHT HALF (Unseen Test Region):")
print(f"  Ground (0):    {class_ious_right[0]:.4f}")
print(f"  Buildings (1): {class_ious_right[1]:.4f}")
print(f"  Trees (2):     {class_ious_right[2]:.4f}")
print(f"  Mean IoU:      {np.mean(class_ious_right):.4f}")

print("\nGeneralization Gap (Left - Right):")
print(f"  Ground:    {class_ious_left[0] - class_ious_right[0]:+.4f}")
print(f"  Buildings: {class_ious_left[1] - class_ious_right[1]:+.4f}")
print(f"  Trees:     {class_ious_left[2] - class_ious_right[2]:+.4f}")
print(f"  Mean:      {np.mean(class_ious_left) - np.mean(class_ious_right):+.4f}")
print("="*60)

#### Interactive Folium Map


In [ ]:
# Stitch left and right predictions together
pred_map_full_image = np.concatenate([pred_map_left, pred_map_right], axis=1)
labels_full_image = np.concatenate([labels_train, labels_test], axis=1)
ndsm_full_image = np.concatenate([ndsm_train, ndsm_test], axis=1)

In [ ]:

# Get bounds for FULL image
with rasterio.open('edinburgh_ndsm.tif') as src:
    full_bounds = src.bounds
    crs = src.crs
    transform = src.transform
    
    # Get corners of full image
    from rasterio.transform import xy
    left_x, bottom_y = xy(transform, src.height, 0)  # Bottom-left
    right_x, top_y = xy(transform, 0, src.width)      # Top-right
    
    # Transform to WGS84
    transformer = Transformer.from_crs(crs, "EPSG:4326", always_xy=True)
    west, south = transformer.transform(left_x, bottom_y)
    east, north = transformer.transform(right_x, top_y)
    
    folium_bounds = [[south, west], [north, east]]
    center_lat = (south + north) / 2
    center_lon = (west + east) / 2

print(f"Map center: ({center_lat:.6f}, {center_lon:.6f})")

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=16,
    tiles=None,
)


# Reconstruct full texture
texture_full_image = np.concatenate([texture_train, texture_test], axis=1)

# Create feature groups
height_group = folium.FeatureGroup(name='🗻 Height (nDSM)', show=False)
texture_group = folium.FeatureGroup(name='📊 Texture (Roughness)', show=False)
truth_group = folium.FeatureGroup(name='✅ Ground Truth', show=True)
pred_group = folium.FeatureGroup(name='🤖 U-Net Predictions', show=True)


# Layer 1: Height (hot colormap)
ndsm_norm = np.clip((ndsm_full_image - 0) / 30, 0, 1)
ndsm_rgb = (plt.cm.hot(ndsm_norm)[:, :, :3] * 255).astype(np.uint8)
folium.raster_layers.ImageOverlay(
    image=ndsm_rgb,
    bounds=folium_bounds,
    opacity=0.8
).add_to(height_group)


# Layer 2: Texture (grayscale)
texture_norm = np.clip((texture_full_image - 0) / 2, 0, 1)
texture_rgb = (plt.cm.gray(texture_norm)[:, :, :3] * 255).astype(np.uint8)
folium.raster_layers.ImageOverlay(
    image=texture_rgb,
    bounds=folium_bounds,
    opacity=0.8
).add_to(texture_group)


# Layer 3: Ground truth (3 classes)
truth_rgba = np.zeros((*labels_full_image.shape, 4), dtype=np.uint8)
truth_rgba[labels_full_image == 0] = [139, 69, 19, 150]   # Brown - Ground
truth_rgba[labels_full_image == 1] = [255, 0, 0, 150]     # Red - Buildings
truth_rgba[labels_full_image == 2] = [0, 255, 0, 150]     # Green - Trees


folium.raster_layers.ImageOverlay(
    image=truth_rgba,
    bounds=folium_bounds
).add_to(truth_group)

# Layer 4: Predictions (3 classes)
pred_rgba = np.zeros((*pred_map_full_image.shape, 4), dtype=np.uint8)
pred_rgba[pred_map_full_image == 0] = [139, 69, 19, 150]   # Brown - Ground
pred_rgba[pred_map_full_image == 1] = [255, 0, 0, 150]     # Red - Buildings
pred_rgba[pred_map_full_image == 2] = [0, 255, 0, 150]     # Green - Trees

folium.raster_layers.ImageOverlay(
    image=pred_rgba,
    bounds=folium_bounds
).add_to(pred_group)

# Add all layers
height_group.add_to(m)
texture_group.add_to(m)
truth_group.add_to(m)
pred_group.add_to(m)

# Layer control
folium.LayerControl(position='topright', collapsed=False).add_to(m)


m